[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Your First Request &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API. Run it first.


In [1]:
import importlib
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
OPEN_METEO = practice_api.open_meteo()
print("ready:", BASE)
print("Open-Meteo:", OPEN_METEO)


ready: http://127.0.0.1:8765
Open-Meteo: https://archive-api.open-meteo.com/v1/archive


**1.** A status code, a header and a field.


In [2]:
response = requests.get(f"{BASE}/stations/svalbard", timeout=10)

print(response.status_code, response.headers["Content-Length"], response.json()["name"])


200 77 Svalbard


Header values are always strings, so `Content-Length` comes back as `'77'`, which `print` shows
without its quotes. Convert it with `int` before doing arithmetic with it.


**2.** The body three ways.


In [3]:
response = requests.get(f"{BASE}/stations/oslo", timeout=10)

for name, value in [("content", response.content), ("text", response.text), ("json()", response.json())]:
    print(f"{name:<8} {type(value).__name__:<6} {value}")


content  bytes  b'{"id": "oslo", "name": "Oslo", "latitude": 59.91, "longitude": 10.75}'
text     str    {"id": "oslo", "name": "Oslo", "latitude": 59.91, "longitude": 10.75}
json()   dict   {'id': 'oslo', 'name': 'Oslo', 'latitude': 59.91, 'longitude': 10.75}


Three types for a single body: bytes as they arrived, a string decoded from them, and a dictionary
parsed from the string.


**3.** The request behind a response.


In [4]:
response = requests.get(f"{BASE}/stations", timeout=10)
sent = response.request

print(sent.method, sent.url, sent.headers["Accept"])


GET http://127.0.0.1:8765/stations */*


`sent.headers` holds the headers requests added, even though no headers were passed to `get`.


**4.** Bergen in Fahrenheit, with `params`.

Weather data by [Open-Meteo.com](https://open-meteo.com/), under
[CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), from the ERA5 reanalysis. Generated using
Copernicus Climate Change Service information 2026.


In [5]:
response = requests.get(OPEN_METEO, timeout=30, params={
    "latitude": 60.39, "longitude": 5.32, "start_date": "2025-01-15", "end_date": "2025-01-17",
    "daily": "temperature_2m_mean", "models": "era5", "temperature_unit": "fahrenheit"})

print(response.url)
print(response.json()["daily_units"]["temperature_2m_mean"], response.json()["daily"]["temperature_2m_mean"])


https://archive-api.open-meteo.com/v1/archive?latitude=60.39&longitude=5.32&start_date=2025-01-15&end_date=2025-01-17&daily=temperature_2m_mean&models=era5&temperature_unit=fahrenheit
°F [46.0, 46.3, 46.4]


requests put the parameters into the query in the order the dictionary listed them, so
`temperature_unit` comes last. The `°F` confirms that Open-Meteo used the unit.


**5.** A redirect, and its history.


In [6]:
response = requests.get(f"{BASE}/v0/stations/bergen", timeout=10)

print("final URL:", response.url)
print("history:  ", [earlier.status_code for earlier in response.history])
print("final:    ", response.status_code)


final URL: http://127.0.0.1:8765/stations/bergen
history:   [301]
final:     200


`history` holds Response objects, so the list comprehension reads the status code out of every
response in it.


**6.** A 404 as an answer, and every other error as a failure.


In [7]:
def station_name(station_id):
    """A station's name, or None when the practice API has no such station."""
    response = requests.get(f"{BASE}/stations/{station_id}", timeout=10)
    if response.status_code == 404:
        return None
    response.raise_for_status()
    return response.json()["name"]


print(station_name("bergen"))
print(station_name("narvik"))


Bergen
None


The order matters. The `404` check comes first, because a missing station is an answer the caller
expects. `raise_for_status` then stops the program for every other error status, such as a `500`,
which this function has no sensible way to answer.


---

&#8592; **Back to:** [Your First Request](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/03-your-first-request.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
